# EfficientAD Teacher Network Training Only

This notebook trains **ONLY** the teacher network using Algorithm 3 from the EfficientAD paper.

**Requirements:**
- PyTorch with CUDA
- torchvision
- tqdm
- ImageNet-100 dataset 

**What you need before running:**
1. Download ImageNet-100 and extract to `./Imagenet-100`

## 1) Install Dependencies

In [ ]:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
# !pip install tqdm

print('Dependencies ready')

## 2) Imports and Device Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from torch.optim import Adam
from torch.amp import autocast, GradScaler
import numpy as np
from tqdm import tqdm
import os
import gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 3) Model Architecture - Teacher Network

In [ ]:
class EfficientAD_PatchDescriptionNetwork(nn.Module):
    def __init__(self, out_channels=384):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 128, kernel_size=4, stride=1, padding=3)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1)
        self.conv2 = nn.Conv2d(128, 256, kernel_size=4, stride=1, padding=3)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1)
        self.conv3 = nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(256, out_channels, kernel_size=4, stride=1, padding=0)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = F.relu(self.conv3(x))
        x = self.conv4(x)
        return x

print('Teacher model architecture ready')

## 4) Feature Extractor (WideResNet101)

In [ ]:
class PretrainedFeatureExtractor(nn.Module):
    """Feature extractor using WideResNet101_2"""
    def __init__(self, out_channels=384):
        super().__init__()
        self.pretrained_model = models.wide_resnet101_2(weights='IMAGENET1K_V1')
        self.out_channels = out_channels

    def forward(self, x):
        # Extract features from layer2 and layer3
        x = self.pretrained_model.conv1(x)
        x = self.pretrained_model.bn1(x)
        x = self.pretrained_model.relu(x)
        x = self.pretrained_model.maxpool(x)
        x = self.pretrained_model.layer1(x)
        pretrained_output1 = self.pretrained_model.layer2(x)
        pretrained_output2 = self.pretrained_model.layer3(pretrained_output1)

        # Interpolate layer3 to match layer2 spatial dimensions
        b, c, h, w = pretrained_output1.shape
        pretrained_output2 = F.interpolate(pretrained_output2, size=(h, w), mode='bilinear', align_corners=False)
        
        # Concatenate layer2 and layer3
        features = torch.cat([pretrained_output1, pretrained_output2], dim=1)  # [B, C1+C2, H, W]
        b, c, h, w = features.shape
        
        # Reshape and apply adaptive pooling to project to out_channels
        features = features.reshape(b, c, h * w)
        features = features.transpose(1, 2)  # [B, H*W, C]
        target_features = F.adaptive_avg_pool1d(features, self.out_channels)  # [B, H*W, out_channels]
        target_features = target_features.transpose(1, 2)  # [B, out_channels, H*W]
        target_features = target_features.reshape(b, self.out_channels, h, w)
        return target_features

print('Feature extractor ready')

## 5) Utility Functions

In [ ]:
def compute_channel_stats(model, dataloader, device, num_batches=None):
    """Compute mu (mean) and sigma (std) for channel normalization"""
    model.eval()
    vals = []
    with torch.no_grad():
        for i, (img, _) in enumerate(dataloader):
            img = img.to(device)
            out = model(img)  # [B, C, H, W]
            vals.append(out.cpu())
            if num_batches and i + 1 >= num_batches:
                break
    
    # Flatten spatial dimensions: [N*H*W, C]
    X = torch.cat([v.view(v.size(0), v.size(1), -1) for v in vals], dim=0)  # [N, C, H*W]
    mu = X.mean(dim=(0, 2))  # [C]
    sigma = X.std(dim=(0, 2))  # [C]
    return mu.to(device), sigma.to(device)


def save_checkpoint(path, teacher, mu, sigma, optimizer, scaler, iteration):
    """Save training checkpoint"""
    checkpoint = {
        'teacher': teacher.state_dict(),
        'mu': mu,
        'sigma': sigma,
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict(),
        'iteration': iteration,
    }
    torch.save(checkpoint, path)


def load_checkpoint(path, teacher, optimizer, scaler, device):
    """Load training checkpoint"""
    checkpoint = torch.load(path, map_location=device)
    teacher.load_state_dict(checkpoint['teacher'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scaler.load_state_dict(checkpoint['scaler'])
    mu = checkpoint['mu'].to(device)
    sigma = checkpoint['sigma'].to(device)
    iteration = checkpoint['iteration']
    return mu, sigma, iteration

print('Utility functions ready')

## 6) Data Loading

In [ ]:
# Create checkpoint directory
SAVE_DIR = './Save_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Checkpoint directory: {SAVE_DIR}')

# Dataset path
DISTILL_DATASET_PATH = './archive/train_combined'

# Transforms for distillation
distill_transforms = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load ImageNet dataset
distill_dataset = datasets.ImageFolder(DISTILL_DATASET_PATH, transform=distill_transforms)
distill_loader = DataLoader(
    distill_dataset,
    batch_size=4,  
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f'Dataset loaded: {len(distill_dataset)} images')
print(f'DataLoader ready (batch_size=16, {len(distill_loader)} batches)')

## 7) Initialize Models

In [ ]:
# Initialize models
teacher = EfficientAD_PatchDescriptionNetwork(384).to(device)
extractor = PretrainedFeatureExtractor(out_channels=384).to(device)
extractor.eval()  # Feature extractor is not trained

print('Teacher and extractor initialized')
print(f'Teacher parameters: {sum(p.numel() for p in teacher.parameters()):,}')
print(f'Extractor parameters: {sum(p.numel() for p in extractor.parameters()):,}')

## 8) Train Teacher Network (Algorithm 3)

In [ ]:
def train_teacher(
    teacher,
    extractor,
    distill_loader,
    iters=60000,
    device='cuda',
    save_dir=None,
    resume=True
):
    """Train teacher network using Algorithm 3 from EfficientAD paper"""
    teacher.to(device)
    extractor.to(device).eval()
    
    optimizer = Adam(teacher.parameters(), lr=1e-4, weight_decay=1e-5)
    scaler = GradScaler()
    
    start_iter = 0
    mu_psi = sigma_psi = None
    ckpt_path = os.path.join(save_dir, 'teacher_checkpoint.pth') if save_dir else None
    
    # Resume from checkpoint if exists
    if resume and ckpt_path and os.path.exists(ckpt_path):
        print(f'Resuming from checkpoint: {ckpt_path}')
        mu_psi, sigma_psi, start_iter = load_checkpoint(ckpt_path, teacher, optimizer, scaler, device)
        print(f'Resumed from iteration {start_iter}')
    
    # Compute mu/sigma statistics if not loaded
    if mu_psi is None:
        print('Computing mu/sigma statistics from 10,000 ImageNet samples...')
        # 10,000 samples ÷ 16 batch_size = 625 batches
        mu_psi, sigma_psi = compute_channel_stats(extractor, distill_loader, device, num_batches=625)
        print(f'Computed mu: {mu_psi.shape}, sigma: {sigma_psi.shape}')
    
    teacher.train()
    data_iter = iter(distill_loader)
    pbar = tqdm(range(start_iter, iters), desc='Training Teacher')
    
    for i in pbar:
        # Get next batch
        try:
            imgs, _ = next(data_iter)
        except StopIteration:
            data_iter = iter(distill_loader)
            imgs, _ = next(data_iter)
        
        imgs = imgs.to(device)
        
        # Extract features with pretrained extractor
        with torch.no_grad():
            # Extractor expects 512x512 input
            psi = extractor(imgs)  # Already 512x512
            psi_norm = (psi - mu_psi[None, :, None, None]) / (sigma_psi[None, :, None, None] + 1e-9)
        
        # Teacher expects 256x256 input
        imgs_256 = F.interpolate(imgs, size=(256, 256), mode='bilinear', align_corners=False)
        
        # Forward pass with autocast
        with autocast(device_type='cuda'):
            teacher_out = teacher(imgs_256)
            loss = ((teacher_out - psi_norm) ** 2).mean()
        
        # Backward pass
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        pbar.set_postfix(loss=f'{loss.item():.6f}')
        
        # Save checkpoint every 1000 iterations
        if ckpt_path and (i + 1) % 1000 == 0:
            save_checkpoint(ckpt_path, teacher, mu_psi, sigma_psi, optimizer, scaler, i + 1)
            print(f'💾 Checkpoint saved at iteration {i+1}')
        
        # Memory cleanup
        if (i + 1) % 100 == 0:
            del psi, psi_norm, teacher_out, loss
            torch.cuda.empty_cache()
            gc.collect()
    
    print('Teacher training complete!')
    return teacher, mu_psi, sigma_psi


print('Training function ready')

## 9) Run Teacher Training

In [ ]:
print(' Starting teacher training...')
print('This will take several hours depending on your GPU')
print()

teacher, mu_psi, sigma_psi = train_teacher(
    teacher=teacher,
    extractor=extractor,
    distill_loader=distill_loader,
    iters=60000,  
    device=device,
    save_dir=SAVE_DIR,
    resume=True
)

print('Teacher training COMPLETE!')  

## 10) Save Trained Teacher

In [ ]:
final_ckpt_path = os.path.join(SAVE_DIR, 'teacher_final.pth')
torch.save({
    'teacher': teacher.state_dict(),
    'mu': mu_psi,
    'sigma': sigma_psi,
}, final_ckpt_path)

print(f'Trained teacher saved to: {final_ckpt_path}')
print(f'Normalization parameters (mu, sigma) also saved')
print()
print('Next steps:')
print('1. Use this teacher for Student + Autoencoder training (Algorithm 1)')
print(f'2. Load from checkpoint: torch.load(\"{final_ckpt_path}\")')

In [ ]:
val_path = './archive/val.X'

if os.path.exists(val_path):
    from torchvision.datasets import ImageFolder
    
    val_transforms = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    val_dataset = ImageFolder(val_path, transform=val_transforms)
    val_loader = DataLoader(
        val_dataset,
        batch_size=4,
        shuffle=False, 
        num_workers=2,
        pin_memory=False
    )
    
    print(f"Validation dataset loaded: {len(val_dataset)} images")


In [ ]:
def validate_teacher_accuracy(teacher, extractor, val_loader, mu_psi, sigma_psi, device, num_batches=None):
    """
    Validate teacher by computing loss on validation set.
    Lower loss = better teacher performance.
    """
    import numpy as np
    from tqdm import tqdm
    
    teacher.eval()
    extractor.eval()
    
    losses = []
    batch_count = 0
    
    print("🔍 Computing validation loss...")
    
    with torch.no_grad():
        for imgs, _ in tqdm(val_loader, desc="Validating"):
            imgs = imgs.to(device)
            
            # Feature extractor output (normalized)
            psi = extractor(imgs)
            psi_norm = (psi - mu_psi[None, :, None, None]) / (sigma_psi[None, :, None, None] + 1e-9)
            
            # Teacher output
            imgs_256 = F.interpolate(imgs, size=(256, 256), mode='bilinear', align_corners=False)
            teacher_out = teacher(imgs_256)
            
            # Compute loss
            loss = ((teacher_out - psi_norm) ** 2).mean()
            losses.append(loss.item())
            
            batch_count += 1
            if num_batches and batch_count >= num_batches:
                break
    
    # Statistics
    avg_loss = np.mean(losses)
    std_loss = np.std(losses)
    min_loss = np.min(losses)
    max_loss = np.max(losses)
    
    print(f"\nValidation Results:")
    print(f"   Batches evaluated: {len(losses)}")
    print(f"   Average Loss: {avg_loss:.6f}")
    print(f"   Std Dev: {std_loss:.6f}")
    print(f"   Min Loss: {min_loss:.6f}")
    print(f"   Max Loss: {max_loss:.6f}")
    
    # Interpretation
    print(f"\n Interpretation:")
    if avg_loss < 0.5:
        print("Excellent! Teacher generalizes very well to unseen data.")
    elif avg_loss < 0.8:
        print("Good! Teacher is learning effectively.")
    elif avg_loss < 1.2:
        print("Acceptable, but could benefit from more training.")
    else:
        print("Poor generalization. Check for issues or train longer.")
    
    teacher.train()
    return {
        'avg_loss': avg_loss,
        'std_loss': std_loss,
        'min_loss': min_loss,
        'max_loss': max_loss,
        'losses': losses
    }


In [ ]:
# Load our teacher
checkpoint = torch.load('./Save_checkpoints/teacher_final.pth', map_location=device)
teacher.load_state_dict(checkpoint['teacher'])
teacher.eval()

# Compute mu/sigma from your data
print("Computing mu/sigma...")
mu_psi, sigma_psi = compute_channel_stats(extractor, distill_loader, device, num_batches=100)

# Validate
val_results = validate_teacher_accuracy(
    teacher, extractor, val_loader,
    mu_psi, sigma_psi, device, num_batches=500
)

print(f"Teacher validation loss: {val_results['avg_loss']:.6f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot histogram of validation losses
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.hist(val_results['losses'], bins=50, alpha=0.7, edgecolor='black')
plt.axvline(val_results['avg_loss'], color='red', linestyle='--', linewidth=2, label=f'Mean: {val_results["avg_loss"]:.3f}')
plt.xlabel('Loss')
plt.ylabel('Frequency')
plt.title('Distribution of Validation Losses')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(val_results['losses'], alpha=0.6)
plt.axhline(val_results['avg_loss'], color='red', linestyle='--', linewidth=2, label=f'Mean: {val_results["avg_loss"]:.3f}')
plt.xlabel('Batch Index')
plt.ylabel('Loss')
plt.title('Validation Loss per Batch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('teacher_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved visualization to teacher_validation.png")
